## Problem *(unicode1)*: Understanding Unicode (1 point)

**(a)** What Unicode character does `chr(0)` return?  
**(Ans)** Null char 

**(b)** How does this character’s string representation (`__repr__()`) differ from its printed representation?  
**(Ans)** `__repr__` -> `'\x00'`; `__str__` -> *(NULL, often invisible)*; `print` uses `__str__`.

**(c)** What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:

```python
>>> chr(0)
>>> print(chr(0))
>>> "this is a test" + chr(0) + "string"
>>> print("this is a test" + chr(0) + "string") 
``` 
**Deliverable:** Where ever print we don’t see the char but its visible when we do direct "" print as it uses `__repr__` (len captures the invisible char)



In [7]:
chr(0)

'\x00'

In [8]:
print(chr(0))

 


In [9]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [10]:
print("this is a test" + chr(0) + "string")
print(len("this is a test" + chr(0) + "string"), len("this is a test" + "string"))

this is a test string
21 20


## Problem *(unicode2)*: Unicode Encodings (3 points)

**(a)** What are some reasons to prefer training our tokenizer on **UTF-8 encoded bytes**, rather than **UTF-16** or **UTF-32**?  
It may be helpful to compare the output of these encodings for various input strings.  

**(Ans)** Utf-32, utf-16 use lot more bytes (2, 4) for encoding ascii chars whereas utf-8 uses just 1 byte, commonly used on web utf-16, utf-32 encoding have lot more zeros, most documents on net already use utf-8


**(b)** Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.

```python
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

>>> decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
'hello'
```
**(Ans)** Utf-8 might be 2 or more bytes long this gives wrong result example; example try to encode and decode é



**(c)** Give a **two-byte sequence** that does **not** decode to any Unicode character(s).

**(Ans)** b"\xC0\x80"



# Problem (train_bpe_tinystories): BPE Training on TinyStories (2 points)

---

### (a)

Train a byte-level BPE tokenizer on the TinyStories dataset, using a maximum vocabulary size of **10,000**.  
Make sure to add the TinyStories `<|endoftext|>` special token to the vocabulary.  
Serialize the resulting vocabulary and merges to disk for further inspection.  

- How many hours and memory did training take?  
- What is the longest token in the vocabulary?  
- Does it make sense?  

**Resource requirements:** ≤ 30 minutes (no GPUs), ≤ 30GB RAM  

**Hint**:  
You should be able to get under 2 minutes for BPE training using `multiprocessing` during pretokenization and the following two facts:  

1. The `<|endoftext|>` token delimits documents in the data files.  
2. The `<|endoftext|>` token is handled as a special case before the BPE merges are applied.  

**Deliverable**:  
Ans) it took 294s (0.08 hrs), 6GB memory to train, the logenst token in vocabulary is ' accomplishment', yes it makes sense

---

### (b)

Profile your code. What part of the tokenizer training process takes the most time?  

**Deliverable**:  
Ans) getting the max freq pair and regex matching take most of the time in the code

# Problem (train_bpe_expts_owt): BPE Training on OpenWebText (2 points)

---

### (a)

Train a byte-level BPE tokenizer on the OpenWebText dataset, using a maximum vocabulary size of **32,000**.  
Serialize the resulting vocabulary and merges to disk for further inspection.  

- What is the longest token in the vocabulary?  
- Does it make sense?  

**Resource requirements:** ≤ 12 hours (no GPUs), ≤ 100GB RAM  

**Deliverable**:  
Ans) it took 8.33 hrs, 51GB memory to train, the logest token in vocabulary is 'ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ'

---

### (b)

Compare and contrast the tokenizer that you get training on TinyStories versus OpenWebText.  

**Deliverable**:  
A) owt has bigger tokens than tiny-stories 

In [11]:
import pstats
base_path = './artifacts/bpe/'
dataset = 'TinyStoriesV2-GPT4/'
filename = 'train-10k.prof'
stats = pstats.Stats(base_path + dataset + filename).strip_dirs().sort_stats("cumulative")
stats.print_stats(40)

Fri Sep 26 08:38:37 2025    ./artifacts/bpe/TinyStoriesV2-GPT4/train-10k.prof

         502669591 function calls (502669473 primitive calls) in 294.064 seconds

   Ordered by: cumulative time
   List reduced from 458 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    2.300    2.300  294.064  294.064 bpe.py:85(train_bpe)
     9769   85.317    0.009  173.580    0.018 {built-in method builtins.max}
        4    0.000    0.000  112.900   28.225 threading.py:611(wait)
        4    0.000    0.000  112.900   28.225 threading.py:295(wait)
       19  112.900    5.942  112.900    5.942 {method 'acquire' of '_thread.lock' objects}
        1    0.000    0.000  112.899  112.899 pool.py:369(starmap)
        1    0.000    0.000  112.898  112.898 pool.py:767(get)
        1    0.000    0.000  112.898  112.898 pool.py:764(wait)
490366278   88.263    0.000   88.263    0.000 bpe.py:126(<lambda>)
   555560    2.130    0.000    2.788    0.000 

In [14]:
import pstats
base_path = './artifacts/bpe/'
dataset = 'owt/'
filename = 'train-32k.prof'
stats = pstats.Stats(base_path + dataset + filename).strip_dirs().sort_stats("cumulative")
stats.print_stats(40)

Fri Sep 26 08:38:37 2025    ./artifacts/bpe/owt/train-32k.prof

         71237019304 function calls (71237019186 primitive calls) in 30017.329 seconds

   Ordered by: cumulative time
   List reduced from 460 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1  433.840  433.840 30017.327 30017.327 bpe.py:85(train_bpe)
    31770 15304.279    0.482 28338.675    0.892 {built-in method builtins.max}
69115890171 13034.396    0.000 13034.396    0.000 bpe.py:126(<lambda>)
 72062904  482.066    0.000  568.504    0.000 bpe.py:55(_get_pair_from_word)
        4    0.000    0.000  225.504   56.376 threading.py:611(wait)
        4    0.000    0.000  225.504   56.376 threading.py:295(wait)
       19  225.504   11.869  225.504   11.869 {method 'acquire' of '_thread.lock' objects}
        1    0.000    0.000  225.504  225.504 pool.py:369(starmap)
        1    0.000    0.000  225.503  225.503 pool.py:767(get)
        1    0.000    0.000  225.5

In [11]:
import gzip, pickle

vocab_path = 'artifacts/bpe/TinyStoriesV2-GPT4/train-10k-vocab.pkl.gz'
merges_path = 'artifacts/bpe/TinyStoriesV2-GPT4/train-10k-merges.pkl.gz'

with gzip.open(vocab_path, 'rb') as g:
    vocab = pickle.load(g)
with gzip.open(merges_path, 'rb') as g:
    merges = pickle.load(g)

print(len(vocab), len(merges))

10000 9743


In [12]:
import gzip, pickle

vocab_path = 'artifacts/bpe/owt/train-32k-vocab.pkl.gz'
merges_path = 'artifacts/bpe/owt/train-32k-merges.pkl.gz'

with gzip.open(vocab_path, 'rb') as g:
    vocab = pickle.load(g)
with gzip.open(merges_path, 'rb') as g:
    merges = pickle.load(g)

print(len(vocab), len(merges))

32000 31743


In [ ]:
import regex as re
END_OF_TEXT = "<|endoftext|>"
EOT_REGEX = re.compile(re.escape(END_OF_TEXT))
def sample_10_docs(path: str) -> list[str]:
    with open(path, 'r') as f:
        data = f.read().split(END_OF_TEXT)

    return data[:10]

In [5]:
sample_10_docs('data/owt-train.txt')


: 